# 16. Does detection masquerade as prediction across methods and event definitions? (development period, exploratory)

In notebook 12, the "caught" count of the V+F model mixed warning **before** the onset (not reproduced) with detection **after** the onset (reproduced). I check that the pattern is not a property of one particular model. Here I apply **the same decomposition and the same strict null models** to four different methods and three event definitions.

I compare four methods. All of them use 2021 for training, 2022 for calibration and a calendar budget of 20 per month and 240 per year; I fix the hyperparameters in advance and do no tuning.
- M0 V+F logistic: the baseline model so far.
- M1 volatility threshold: no training; it alerts when z_vol (30-day z-score of 32-minute realized volatility) exceeds a threshold.
- M2 VPIN: the volume-synchronized order-imbalance measure that Easley, López de Prado and O'Hara (2012) proposed as an early warning of the flash crash. I split buy and sell volume directly with taker buy volume; the bucket size is 1/50 of the trailing 30-day mean daily volume and the window is 50 buckets (the original paper's default). Every value uses only data up to that minute.
- M3 gradient boosting: a richer set of causal features (17, including multi-horizon returns and realized volatility, drawdown from the peak, volume intensity and sell ratio).

An event is a decline of **3%, 4% or 5%** from the previous 240-minute high (5-minute confirmation, 24-hour spacing).

As in notebook 12, I separate pre-onset warning, in-progress detection (after the onset, at least 30 minutes before the crash), the composite (reference) and onset-anchored warning (secondary). I use three nulls: the circular shift (7 days or more), the short shift (4 hours to 1 day) and the regime-matched null (same volatility decile, ±30 days).

I fixed an illusion rule before seeing results and recorded it in `METHODS_SPEC.json`: if the composite is significant under the primary null (p<0.05), pre-onset warning is not significant under the regime-matched null (p≥0.05) and in-progress detection is significant under it, I read the result as "detection masquerading as prediction".

This analysis is exploratory, although it is the first time I evaluate M1–M3 on 2023–2026. I save the thresholds and models from this notebook with hashes to `METHODS_FROZEN.json`; notebook 17 uses them unchanged in its one-shot test on the unused 2018–2019 multi-coin period. I write the results to `methods_definitions_v1`. This saved copy keeps the output of only one cell; `METHODS_SUMMARY.json` and the tables in that folder hold all results.

In [ ]:
from pathlib import Path
from collections import deque, defaultdict
import io, os, sys, json, math, time, hashlib, zipfile, warnings, urllib.request, urllib.error
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])
try:
    from numba import njit
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: paths below must point to a folder with the same structure.')

DRIVE = Path('/content/drive/MyDrive')
SPOT_BASE = DRIVE / 'btc_lead_cost_results_77months_spot'      # spot results (notebooks 01–07)
FUT_BASE = DRIVE / 'btc_lead_cost_results_77months'            # futures results, read only
BINANCE = 'https://data.binance.vision/data/'
NS = 60 * 10**9
LEAD_MIN = 30; LEAD_MAX = 240; COOLDOWN = 240; COOLDOWN_MIN = 240
B = 240; MONTH_CAP = 20
EVAL_START = '2023-01-01'; EVAL_END = '2026-05-01'

_src = SPOT_BASE / 'cache' / 'DATA_SOURCE.json'
if not _src.is_file() or json.loads(_src.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'Spot cache not found: {_src}. Run notebook 01 first.')
REF_PATH = SPOT_BASE / 'causal_alert_selection_v1' / 'spot_reference.json'
THR_PATH = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'spot_frozen_thresholds.json'
for _p in (REF_PATH, THR_PATH):
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 05 and 06 are required: {_p}')
REF = json.loads(REF_PATH.read_text())
FROZEN = json.loads(THR_PATH.read_text())
assert REF.get('venue') == 'spot' and FROZEN.get('venue') == 'spot'
TH_CAL = float(REF['TH_CAL']); TH_HIGH = float(REF['TH_HIGH']); TH_BUCKET = float(FROZEN['TH_BUCKET'])
print('SPOT cache verified. Frozen spot thresholds:', dict(TH_CAL=TH_CAL, TH_HIGH=TH_HIGH, TH_BUCKET=TH_BUCKET))


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def load_btc_spot():
    px = pd.read_parquet(SPOT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet').sort_index()
    sg = pd.read_parquet(SPOT_BASE / 'cache' / 'btc_signals_1m.parquet', columns=['z_vol', 'tsr', 'z_hs']).sort_index()
    for fr in (px, sg):
        assert isinstance(fr.index, pd.DatetimeIndex) and fr.index.tz is not None
        fr.index = fr.index.tz_convert('UTC')
    assert px.index.equals(sg.index), 'price/signal caches must share one 1-minute index'
    assert px.index.is_unique and (np.diff(px.index.asi8) == NS).all()
    return px, sg


def load_catalog(path):
    E = pd.read_csv(path)
    for c in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
        E[c] = pd.to_datetime(E[c], utc=True)
    E = E.sort_values('breach_utc').reset_index(drop=True)
    assert E.event_id.is_unique
    return E


def label_from_catalog(T, E):
    """1 if a 4% breach occurs 30-240 minutes after minute t (same rule as notebooks 04–07)."""
    b = E.breach_utc.astype('int64').to_numpy()
    pos = np.searchsorted(b, T + LEAD_MIN * NS, 'left')
    y = np.zeros(len(T), dtype=np.int8)
    ok = pos < len(b)
    y[ok] = (b[pos[ok]] <= T[ok] + LEAD_MAX * NS).astype(np.int8)
    return y


def ts(s):
    return pd.Timestamp(s, tz='UTC')


def in_period(T, start, end):
    return (T >= ts(start).value) & (T < ts(end).value)


def train_mask(T, valid, year=2021):
    return valid & (T >= ts(f'{year}-01-01').value) & (T < ts(f'{year + 1}-01-01').value - (LEAD_MAX + 5) * NS)


def new_logit():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400,
                                                              solver='lbfgs', random_state=17))


def events_between(E, start, end):
    """Evaluation events: breach in [start+4h, end) so every warning window lies inside the period."""
    return E[(E.breach_utc >= ts(start) + pd.Timedelta(minutes=LEAD_MAX)) & (E.breach_utc < ts(end))].reset_index(drop=True)


# --- emit_calendar, as in notebook 05 ---
def emit_calendar(t,eligible,B=240,cap=20,cooldown_min=240):
    t=np.asarray(t,dtype=np.int64)
    eligible=np.asarray(eligible,dtype=bool)
    assert len(t)==len(eligible) and np.all(np.diff(t)>0)
    ids=np.flatnonzero(eligible)
    z=[]; mcount=0; ycount=0; pm=None; py=None; last=-(10**20)
    for k in ids:
        ti=int(t[k]); dt=pd.Timestamp(ti,tz='UTC')
        year=int(dt.year); month=year*12+int(dt.month)
        if year!=py: ycount=0; py=year
        if month!=pm: mcount=0; pm=month
        if ycount>=B or mcount>=cap or ti-last<cooldown_min*NS:continue
        z.append(ti); last=ti; ycount+=1; mcount+=1
    return np.asarray(z,dtype=np.int64)

# --- match_events, as in notebook 05 ---
def match_events(events,alerts):
    alerts=np.sort(np.asarray(alerts,dtype=np.int64))
    out=[];used=set()
    for ev in events.itertuples(index=False):
        breach=ev.breach_utc.value; onset=ev.onset_1pct_utc.value
        left=int(np.searchsorted(alerts,breach-LEAD_MAX*NS,'left'))
        right=int(np.searchsorted(alerts,breach-LEAD_MIN*NS,'right'))
        for j in range(left,right):
            if j not in used:
                used.add(j)
                out.append(dict(event_id=int(ev.event_id),breach_utc=ev.breach_utc,
                                alarm_utc=pd.Timestamp(int(alerts[j]),tz='UTC'),
                                lead_min=(breach-int(alerts[j]))/NS,
                                pre1pct=bool(alerts[j]<onset)))
                break
    return pd.DataFrame(out,columns=['event_id','breach_utc','alarm_utc','lead_min','pre1pct'])

# --- statistics, as in notebook 05 ---
def statistics(events,alerts):
    hits=match_events(events,alerts)
    n=len(alerts);N=len(events)
    return dict(alarms=n,caught=len(hits),before_1pct=int(hits.pre1pct.sum()),
                false_alarms=n-len(hits),recall=len(hits)/N,
                pre_onset_recall=int(hits.pre1pct.sum())/N,
                alert_precision=(len(hits)/n if n else 0))



def cal_fields(t):
    idx = pd.to_datetime(np.asarray(t, dtype='int64'), utc=True)
    years = idx.year.to_numpy().astype(np.int64)
    months = (idx.year.to_numpy() * 12 + idx.month.to_numpy()).astype(np.int64)
    return years, months


@njit(cache=False)
def _calendar_core(t, elig, years, months, B, caps, cooldown_ns):
    # Line-by-line port of emit_calendar with a per-minute monthly cap (caps) so a surge policy can reuse it.
    out = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B or mcount >= caps[i] or ti - last < cooldown_ns:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1; mcount += 1
    return out[:k].copy()


_CAL_CACHE = [None, None]


def emit_calendar_fast(t, eligible, B=240, cap=20, cooldown_min=240, caps=None):
    t = np.asarray(t, dtype=np.int64); eligible = np.asarray(eligible, dtype=bool)
    assert len(t) == len(eligible) and (len(t) < 2 or np.all(np.diff(t) > 0))
    if _CAL_CACHE[0] is t:
        years, months = _CAL_CACHE[1]
    else:
        years, months = cal_fields(t); _CAL_CACHE[0] = t; _CAL_CACHE[1] = (years, months)
    if caps is None:
        caps = np.full(len(t), int(cap), np.int64)
    return _calendar_core(t, eligible, years, months, int(B), np.asarray(caps, np.int64), np.int64(cooldown_min * NS))


def check_fast_calendar(t, s):
    """Stop if the compiled calendar ever differs from the notebook 05 reference implementation."""
    qs = np.unique(np.quantile(s[np.isfinite(s)], [0., .5, .8, .9, .95, .99, .999]))
    for th in qs:
        e = np.isfinite(s) & (s >= th)
        for B_, cap_ in [(240, 20), (60, 5)]:
            if not np.array_equal(emit_calendar(t, e, B=B_, cap=cap_), emit_calendar_fast(t, e, B=B_, cap=cap_)):
                raise AssertionError(f'fast calendar differs from the reference at th={th}, B={B_}')
    return len(qs) * 2


def score_grid(sc):
    Q = np.unique(np.r_[0., np.linspace(.10, .995, 33), .997, .999, .9999, 1.])
    return np.unique(np.r_[np.nextafter(sc.min(), -np.inf), np.quantile(sc, Q), np.nextafter(sc.max(), np.inf)])


def calibrate_calendar(t_cal, s_cal, grid=None, B_=240, cap_=20):
    """Notebook 05 rule: the most 2022 alerts reachable (cap = B), then the fewest raw candidates (strictest)."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        e = s_cal >= th
        rows.append((float(th), len(emit_calendar_fast(t_cal, e, B=B_, cap=cap_)), int(e.sum())))
    df = pd.DataFrame(rows, columns=['threshold', 'cal_alerts', 'raw_candidates'])
    best = df[df.cal_alerts == df.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(best.threshold), int(best.cal_alerts), df


def mcnemar_one_sided(a_only, b_only):
    """P(at least b_only successes | n discordant, p=.5): evidence that B beats A."""
    n = a_only + b_only
    return float(binomtest(b_only, n, .5, alternative='greater').pvalue) if n else 1.0


def mcnemar_two_sided(a_only, b_only):
    n = a_only + b_only
    return float(binomtest(min(a_only, b_only), n, .5).pvalue) if n else 1.0


def paired(events, base_alerts, new_alerts):
    mb = match_events(events, base_alerts); mn = match_events(events, new_alerts)
    hb, hn = set(mb.event_id), set(mn.event_id)
    pb, pn = set(mb.loc[mb.pre1pct, 'event_id']), set(mn.loc[mn.pre1pct, 'event_id'])
    return dict(base_caught=len(hb), new_caught=len(hn), base_only_caught=len(hb - hn), new_only_caught=len(hn - hb),
                p_caught_two_sided=mcnemar_two_sided(len(hb - hn), len(hn - hb)),
                base_pre1pct=len(pb), new_pre1pct=len(pn), base_only_pre1pct=len(pb - pn), new_only_pre1pct=len(pn - pb),
                p_pre1pct_two_sided=mcnemar_two_sided(len(pb - pn), len(pn - pb)),
                p_pre1pct_one_sided_new_better=mcnemar_one_sided(len(pb - pn), len(pn - pb)))
RAW_COLS = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume',
            'trade_count', 'taker_buy_vol', 'taker_buy_quote', 'ignore']


def month_list(first, last):
    return [p.strftime('%Y-%m') for p in pd.period_range(first, last, freq='M')]


def _fetch(url, tries=5):
    last = None
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            last = e
        except Exception as e:  # network hiccup: retry
            last = e
        time.sleep(2 ** k)
    raise RuntimeError(f'download failed: {url} ({last})')


def get_archives(prefix, symbol, months, dest, allow_missing=False):
    """Binance public monthly 1m klines (prefix 'spot' or 'futures/um'), SHA-256 checked against .CHECKSUM.
    Already-downloaded files whose hash matches are reused, so an interrupted run resumes."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    rows = []
    for ym in months:
        url = f'{BINANCE}{prefix}/monthly/klines/{symbol}/1m/{symbol}-1m-{ym}.zip'
        z = dest / f'{symbol}-1m-{ym}.zip'
        chk = _fetch(url + '.CHECKSUM')
        if chk is None:
            if allow_missing:
                rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=None, status='not_listed'))
                continue
            raise RuntimeError(f'archive not published: {url}')
        expected = chk.decode().split()[0].strip().lower()
        if z.is_file() and sha256_file(z) == expected:
            status = 'cached'
        else:
            data = _fetch(url)
            if data is None:
                raise RuntimeError(f'archive missing although checksum exists: {url}')
            if hashlib.sha256(data).hexdigest() != expected:
                raise RuntimeError(f'SHA-256 mismatch: {url}')
            part = z.parent / (z.name + '.part'); part.write_bytes(data); part.replace(z)
            status = 'downloaded'
        rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=expected, status=status))
    man = pd.DataFrame(rows)
    print(symbol, prefix, man.status.value_counts().to_dict(), flush=True)
    return man


def parse_kline_zip(path):
    """Header or no header; ms or microsecond timestamps (spot archives switched to microseconds in 2025)."""
    with zipfile.ZipFile(path) as zf:
        members = [m for m in zf.namelist() if m.lower().endswith('.csv')]
        if len(members) != 1:
            raise ValueError(f'expected one CSV in {path}')
        with zf.open(members[0]) as fd:
            x = pd.read_csv(fd, header=None, low_memory=False)
    if str(x.iloc[0, 0]).strip().lower() in ('open_time', 'timestamp'):
        x = x.iloc[1:].reset_index(drop=True)
    if x.shape[1] < 12:
        raise ValueError(f'unexpected kline schema in {path}: {x.shape[1]} columns')
    x = x.iloc[:, :12]; x.columns = RAW_COLS
    number = pd.to_numeric(x.open_time, errors='raise')
    mag = float(number.abs().median())
    unit = 'us' if mag > 1e14 else ('ms' if mag > 1e11 else 's')
    z = x[['open', 'high', 'low', 'close', 'volume', 'taker_buy_vol']].apply(pd.to_numeric, errors='coerce')
    z.index = pd.DatetimeIndex(pd.to_datetime(number, unit=unit, utc=True), name='open_time')
    return z[~z.index.duplicated(keep='last')].sort_index().astype('float64')


def load_archives(manifest, folder):
    parts = [parse_kline_zip(Path(folder) / f"{r.symbol}-1m-{r.month}.zip") for r in manifest.itertuples() if r.status != 'not_listed']
    raw = pd.concat(parts).sort_index(kind='stable')
    return raw[~raw.index.duplicated(keep='last')]


def regularize(raw, start=None, end=None):
    """Notebook 01 gap policy: complete UTC minute grid; missing runs of <=5 minutes get forward-filled OHLC and
    zero volume; longer gaps stay NaN."""
    full = pd.date_range(raw.index.min() if start is None else start, raw.index.max() if end is None else end,
                         freq='min', tz='UTC')
    out = raw.reindex(full); out.index.name = 'open_time'
    missing = ~full.isin(raw.index)
    if missing.any():
        changes = np.r_[True, missing[1:] != missing[:-1]]
        grp = np.cumsum(changes) - 1
        runs = np.bincount(grp[missing], minlength=int(grp.max()) + 1)
        short = missing & (runs[grp] <= 5)
        for c in ['open', 'high', 'low', 'close']:
            out.loc[short, c] = out[c].ffill().loc[short]
        for c in ['volume', 'taker_buy_vol']:
            out.loc[short, c] = 0.0
    return out.astype('float64')


# --- chunked_hs, as in notebook 01 ---
def chunked_hs(close,M=32,K=8,chunk=100_000):
    p=close.to_numpy(dtype=np.float32)
    result=np.full(len(p),np.nan,dtype=np.float64)
    for start in range(M-1,len(p),chunk):
        end=min(len(p),start+chunk)
        w=sliding_window_view(p[start-M+1:end],M)
        zero_mean=w-w.mean(axis=1,keepdims=True)
        result[start:end]=np.abs(np.fft.rfft(zero_mean,axis=1))[:,-K:].mean(axis=1)
        if start==(M-1) or end==len(p): print(f'FFT computation {end:,}/{len(p):,}',flush=True)
    return pd.Series(result,index=close.index)



def compute_spot_features(px, with_hs=True):
    """Exactly the notebook 01 signal formulas (z_hs, z_vol, tsr), float32 like the cache."""
    close = px['close']
    r = np.log(close).diff()
    M = 32; rolling_min = 30 * 1440
    out = {}
    if with_hs:
        hs = chunked_hs(close, M=32, K=8)
        log_hs = np.log1p(hs)
        hs_roll = log_hs.rolling(rolling_min, min_periods=rolling_min // 2)
        out['z_hs'] = ((log_hs - hs_roll.mean().shift(1)) / hs_roll.std().shift(1).replace(0, np.nan)).astype('float32')
    rv = r.rolling(M - 1, min_periods=M - 1).std()
    lrv = np.log(rv.where(rv > 0))
    rv_base = lrv.rolling(rolling_min, min_periods=rolling_min // 2)
    out['z_vol'] = ((lrv - rv_base.mean().shift(1)) / rv_base.std().shift(1).replace(0, np.nan)).astype('float32')
    out['tsr'] = ((px.volume - px.taker_buy_vol) / px.volume.replace(0, np.nan)).clip(0, 1).astype('float32')
    return pd.DataFrame(out, index=px.index)


# --- price_only_catalog, as in notebook 07 ---
def price_only_catalog(close,volume,index,H=240,decline=.04,sustain=5,refractory=1440):
    c=np.asarray(close,dtype=np.float64)
    obs=np.isfinite(c)&(c>0)&np.isfinite(volume)&(np.asarray(volume)>0)
    prior=pd.Series(np.where(obs,c,np.nan)).rolling(H,min_periods=H).max().shift(1).to_numpy()
    raw=obs & np.isfinite(prior) & (c<=(1-decline)*prior)
    fresh=raw & ~np.r_[False,raw[:-1]]
    rows=[];last_confirm=-10**12
    for i in np.flatnonzero(fresh):
        if i+sustain>len(c) or i-last_confirm<refractory:continue
        threshold=(1-decline)*prior[i]
        if not(obs[i:i+sustain].all() and np.all(c[i:i+sustain]<=threshold)):continue
        if i-H<0 or not obs[i-H:i].all():continue
        prev=c[i-H:i];peak_i=int(i-H+np.argmax(prev))
        if peak_i<=last_confirm:continue
        onset_candidates=np.flatnonzero(c[peak_i+1:i+1]<=.99*c[peak_i])
        if not len(onset_candidates):continue
        oi=int(peak_i+1+onset_candidates[0])
        rows.append({'event_id':len(rows)+1,'peak_utc':index[peak_i],
                     'onset_1pct_utc':index[oi], 'breach_utc':index[i],
                     'confirmed_utc':index[i+sustain-1],
                     'peak_close':float(c[peak_i]),'breach_close':float(c[i]),
                     'decline_at_breach_pct':(c[i]/c[peak_i]-1)*100,
                     'peak_to_breach_minutes':i-peak_i,
                     'onset_to_breach_minutes':i-oi})
        last_confirm=i+sustain-1
    df=pd.DataFrame(rows)
    if not len(df):raise RuntimeError('No price-only crash events generated; inspect price history')
    df['year']=pd.DatetimeIndex(df.breach_utc).year
    return df

import joblib
from sklearn.ensemble import HistGradientBoostingClassifier

OUT = SPOT_BASE / 'methods_definitions_v1'; (OUT / 'models').mkdir(parents=True, exist_ok=True)
CAS = SPOT_BASE / 'causal_alert_selection_v1'
DEFS = {'3pct': 0.03, '4pct': 0.04, '5pct': 0.05}
N_NULL = 2000; SEED = 20261030; ALPHA = 0.05
BANDS = {'circular >=7d (primary)': None, 'circular 4h-1d': (240, 1440)}
REGIME_WINDOW_D = 30; RV_WINDOW = 1440; VPIN_BUCKETS_PER_DAY = 50; VPIN_WINDOW = 50; VPIN_LOOKBACK_D = 30
GBM_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=200, l2_regularization=1.0,
                  class_weight='balanced', early_stopping=False, random_state=17)
FEATS = ['z_vol', 'tsr', 'z_hs', 'ret_5', 'ret_15', 'ret_60', 'ret_240', 'rv_15', 'rv_60', 'rv_240', 'rv_ratio_15_240',
         'dd_240', 'dd_1440', 'vol_intensity_15', 'sell_ratio_15', 'sell_ratio_60', 'vpin']
METHODS = {'M0_VF_logit': dict(kind='logit', cols=['z_vol', 'tsr']),
           'M1_vol_threshold': dict(kind='raw', col='z_vol'),
           'M2_VPIN': dict(kind='raw', col='vpin'),
           'M3_GBM_rich': dict(kind='gbm', cols=FEATS)}
KEYS = ['pre', 'post', 'caught', 'onset_anchored']
SPEC = {'title': 'Detection vs prediction across methods and event definitions (dev 2023-01..2026-04, exploratory)',
        'methods': {k: {kk: vv for kk, vv in v.items()} for k, v in METHODS.items()},
        'vpin': dict(reference='Easley, Lopez de Prado, O\'Hara (2012), Review of Financial Studies', buy_volume='taker buy volume',
                     bucket_size=f'trailing {VPIN_LOOKBACK_D}-day mean daily volume / {VPIN_BUCKETS_PER_DAY}', window_buckets=VPIN_WINDOW,
                     minute_granularity='whole minutes fill buckets; value at a minute = VPIN of the last completed bucket'),
        'gbm_params': GBM_PARAMS, 'gbm_features': FEATS,
        'definitions': DEFS, 'training': '2021 minus last 245 min; label = breach of the same definition 30-240 min ahead',
        'calibration': '2022, notebook-5 calendar rule (most alerts reachable, then strictest), 20/month, 240/yr, 240-min cooldown',
        'endpoints': KEYS, 'nulls': list(BANDS) + [f'regime-matched (same 24h-RV decile, +-{REGIME_WINDOW_D}d)'], 'draws_per_null': N_NULL,
        'illusion_rule': 'composite p<0.05 under the primary null AND pre-onset p>=0.05 under the regime-matched null AND post-onset p<0.05 under the regime-matched null',
        'frozen_for_holdout': 'thresholds, logit coefficients and GBM model files (sha256) saved to METHODS_FROZEN.json'}
_raw = (json.dumps(SPEC, indent=2, sort_keys=True, default=str) + '\n').encode()
_sp = OUT / 'METHODS_SPEC.json'
if _sp.exists() and _sp.read_bytes() != _raw:
    _sp.rename(OUT / f'METHODS_SPEC.replaced_{pd.Timestamp.now(tz="UTC"):%Y%m%dT%H%M%S}.json')
_sp.write_bytes(_raw); SPEC_SHA = hashlib.sha256(_raw).hexdigest()
print('Output:', OUT, '| spec sha256', SPEC_SHA[:16])

## 1. Features and VPIN (all causal)

In [ ]:
@njit(cache=False)
def _vpin_core(vol, buy, bucket, n_win):
    out = np.full(len(vol), np.nan); ring = np.zeros(n_win); k = 0; run = 0.0; cv = 0.0; cb = 0.0
    for i in range(len(vol)):
        v = vol[i]; b = buy[i]
        if not (v == v) or not (b == b) or v < 0:
            v = 0.0; b = 0.0
        cv += v; cb += min(max(b, 0.0), v)
        bs = bucket[i]
        if bs == bs and bs > 0 and cv >= bs:
            imb = abs(2.0 * cb - cv) / cv
            j = k % n_win; run += imb - ring[j]; ring[j] = imb; k += 1; cv = 0.0; cb = 0.0
        if k >= n_win:
            out[i] = run / n_win
    return out


def vpin_series(px):
    vol = px.volume.to_numpy(float); buy = px.taker_buy_vol.to_numpy(float)
    daily = px.volume.groupby(px.index.floor('D')).sum()
    size = (daily.rolling(VPIN_LOOKBACK_D, min_periods=VPIN_LOOKBACK_D).mean().shift(1) / VPIN_BUCKETS_PER_DAY)
    bucket = size.reindex(px.index.floor('D')).to_numpy(float)
    return _vpin_core(vol, buy, bucket, VPIN_WINDOW)


def causal_features(px, sg):
    c = px.close.astype(float); lc = np.log(c); r = lc.diff(); vol = px.volume.astype(float); sell = vol - px.taker_buy_vol.astype(float)
    F = pd.DataFrame(index=px.index)
    F['z_vol'] = sg.z_vol.astype(float); F['tsr'] = sg.tsr.astype(float); F['z_hs'] = sg.z_hs.astype(float)
    for k in (5, 15, 60, 240):
        F[f'ret_{k}'] = lc - lc.shift(k)
    for k in (15, 60, 240):
        F[f'rv_{k}'] = r.rolling(k, min_periods=k // 2).std()
    F['rv_ratio_15_240'] = F.rv_15 / F.rv_240
    F['dd_240'] = 1 - c / c.rolling(240, min_periods=120).max()
    F['dd_1440'] = 1 - c / c.rolling(1440, min_periods=720).max()
    F['vol_intensity_15'] = vol.rolling(15, min_periods=8).sum() / (vol.rolling(1440, min_periods=720).sum() / 96)
    F['sell_ratio_15'] = sell.rolling(15, min_periods=8).sum() / vol.rolling(15, min_periods=8).sum()
    F['sell_ratio_60'] = sell.rolling(60, min_periods=30).sum() / vol.rolling(60, min_periods=30).sum()
    F['vpin'] = vpin_series(px)
    return F.replace([np.inf, -np.inf], np.nan).astype('float32')


t_ = time.time()
px, sg = load_btc_spot()
F = causal_features(px, sg)
T = px.index.asi8; close = px.close.to_numpy(float); volume = px.volume.to_numpy(float)
v_ = sg.z_vol.to_numpy(float); f_ = sg.tsr.to_numpy(float); h_ = sg.z_hs.to_numpy(float)
VALID = (np.isfinite(close) & (close > 0) & np.isfinite(volume) & (volume > 0) & np.isfinite(v_) & np.isfinite(f_) & np.isfinite(h_) & (f_ >= 0) & (f_ <= 1))
CAL = VALID & in_period(T, '2022-01-01', '2023-01-01'); EV = VALID & in_period(T, EVAL_START, EVAL_END)
print(f'features ready in {time.time() - t_:.0f}s; VPIN finite share in evaluation: {np.isfinite(F.vpin.to_numpy()[EV]).mean():.4f}')
display(F[EV].describe().T[['mean', 'std', 'min', 'max']].round(4))

## 2. Catalogs by event definition

In [ ]:
TCOLS = ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']
E_ref = load_catalog(SPOT_BASE / 'event_definition_eligibility_audit' / 'price_only_event_catalog.csv')
CATS = {}
for dname, d in DEFS.items():
    E = price_only_catalog(close, volume, px.index, decline=d)
    for c_ in TCOLS:
        E[c_] = pd.to_datetime(E[c_], utc=True)
    E = E.sort_values('breach_utc').reset_index(drop=True)
    if dname == '4pct' and (len(E) != len(E_ref) or any(not np.array_equal(E[c_].astype('int64').to_numpy(), E_ref[c_].astype('int64').to_numpy()) for c_ in TCOLS)):
        raise RuntimeError('GUARD FAIL: 4% catalog differs from the notebook-2 catalog')
    ev = events_between(E, EVAL_START, EVAL_END)
    ev['gap_min'] = (ev.breach_utc - ev.onset_1pct_utc).dt.total_seconds() / 60
    CATS[dname] = dict(E=E, EV=ev)
ED = pd.DataFrame([dict(definition=k, events_all=len(c['E']), events_dev=len(c['EV']), median_onset_to_breach_min=float(c['EV'].gap_min.median()),
                        gap_lt30=int((c['EV'].gap_min < LEAD_MIN).sum()), gap_gt240=int((c['EV'].gap_min > LEAD_MAX).sum())) for k, c in CATS.items()])
ED.to_csv(OUT / '2_events_by_definition.csv', index=False); display(ED)

,definition,events_all,events_dev,median_onset_to_breach_min,gap_lt30,gap_gt240
0,3pct,518,183,113.0,22,0
1,4pct,269,83,119.0,8,0
2,5pct,162,42,128.0,4,0


## 3. Method × definition: training, 2022 calibration, development-period alerts

In [ ]:
X_all = F[FEATS].to_numpy(np.float32)
RUN = {}; frozen = {'spec_sha256': SPEC_SHA, 'configs': {}}
for dname, cat in CATS.items():
    y = label_from_catalog(T, cat['E']); tr = train_mask(T, VALID)
    for mname, m in METHODS.items():
        t0 = time.time(); info = {}
        if m['kind'] == 'raw':
            S = np.where(VALID, F[m['col']].to_numpy(float), np.nan)
        elif m['kind'] == 'logit':
            Xl = F[m['cols']].to_numpy(float); okl = tr & np.isfinite(Xl).all(axis=1)
            mdl = new_logit().fit(Xl[okl], y[okl]); S = np.full(len(T), np.nan)
            okp = VALID & np.isfinite(Xl).all(axis=1); S[okp] = mdl.predict_proba(Xl[okp])[:, 1]
            info = dict(coef=mdl[-1].coef_.ravel().tolist(), intercept=float(mdl[-1].intercept_[0]),
                        scaler_mean=mdl[0].mean_.tolist(), scaler_scale=mdl[0].scale_.tolist())
        else:
            mdl = HistGradientBoostingClassifier(**GBM_PARAMS).fit(X_all[tr], y[tr])
            S = np.full(len(T), np.nan); S[VALID] = mdl.predict_proba(X_all[VALID])[:, 1]
            mp = OUT / 'models' / f'{mname}_{dname}.joblib'; joblib.dump(mdl, mp)
            info = dict(model_file=str(mp.relative_to(SPOT_BASE)), model_sha256=sha256_file(mp))
        okc = CAL & np.isfinite(S); oke = EV & np.isfinite(S)
        th, ncal, _ = calibrate_calendar(T[okc], S[okc])
        a = emit_calendar_fast(T[oke], S[oke] >= th)
        RUN[(mname, dname)] = a
        pd.DataFrame({'alarm_utc': pd.to_datetime(a, utc=True)}).to_csv(OUT / f'alarms_{mname}_{dname}.csv', index=False)
        frozen['configs'][f'{mname}|{dname}'] = dict(method=mname, definition=dname, threshold_2022=th, cal_alerts_2022=ncal,
                                                     dev_alerts=len(a), **info)
        if (mname, dname) == ('M0_VF_logit', '4pct'):
            st = statistics(cat['EV'], a)
            if abs(th - TH_CAL) > 1e-12 or [st['alarms'], st['caught'], st['before_1pct']] != REF['first_eligible']:
                raise RuntimeError(f'GUARD FAIL: M0 at 4% does not reproduce notebook 5 ({th}, {st})')
            print('PASS: M0 at 4% reproduces notebook 5', REF['first_eligible'])
        print(f'{mname:18s} {dname}: threshold {th:.6g}, 2022 alerts {ncal}, dev alerts {len(a)}  ({time.time() - t0:.0f}s)')
    del y
(OUT / 'METHODS_FROZEN.json').write_text(json.dumps(frozen, indent=2, default=float))
print('METHODS_FROZEN.json sha256:', sha256_file(OUT / 'METHODS_FROZEN.json'))

## 4. Endpoint separation and null models (same method as notebook 12)

In [ ]:
def onset_anchored(ev, a):
    used = np.zeros(len(a), bool); n = 0
    for o in np.sort(ev.onset_1pct_utc.astype('int64').to_numpy()):
        l_ = np.searchsorted(a, o - LEAD_MAX * NS, 'left'); r_ = np.searchsorted(a, o, 'left')
        for j in range(l_, r_):
            if not used[j]:
                used[j] = True; n += 1; break
    return n


def counts(ev, a):
    a = np.sort(np.asarray(a, np.int64)); m = match_events(ev, a)
    pre = int(m.pre1pct.sum()) if len(m) else 0
    return dict(pre=pre, post=len(m) - pre, caught=len(m), onset_anchored=onset_anchored(ev, a))


t0_ = ts(EVAL_START).value; L_ = (ts(EVAL_END).value - t0_) // NS


def circ_null(ev, a, n, seed, band=None):
    a = np.sort(a); rng = np.random.default_rng(seed)
    if band is None:
        d = rng.integers(7 * 1440, L_ - 7 * 1440, size=n)
    else:
        d = rng.integers(band[0], band[1] + 1, size=n) * (rng.integers(0, 2, size=n) * 2 - 1)
    out = {k: np.empty(n, int) for k in KEYS}
    for i, dl in enumerate(d):
        c = counts(ev, t0_ + (a - t0_ + int(dl) * NS) % (L_ * NS))
        for k in KEYS:
            out[k][i] = c[k]
    return out


rv = pd.Series(np.log(close)).diff().rolling(RV_WINDOW, min_periods=RV_WINDOW // 2).std().to_numpy()
okd = in_period(T, EVAL_START, EVAL_END) & np.isfinite(rv)
TT = T[okd]; DD = np.minimum((pd.Series(rv[okd]).rank(pct=True).to_numpy() * 10).astype(int), 9)
_order = np.lexsort((TT, DD)); PP = TT[_order]; DS = DD[_order]
STARTS = np.searchsorted(DS, np.arange(10), 'left'); ENDS = np.searchsorted(DS, np.arange(10), 'right')


def regime_null(ev, a, n, seed):
    a = np.sort(a); j = np.clip(np.searchsorted(TT, a, 'right') - 1, 0, len(TT) - 1); ad = DD[j]; W = REGIME_WINDOW_D * 1440 * NS
    lo = np.array([STARTS[q] + np.searchsorted(PP[STARTS[q]:ENDS[q]], ti - W, 'left') for ti, q in zip(a, ad)])
    hi = np.array([STARTS[q] + np.searchsorted(PP[STARTS[q]:ENDS[q]], ti + W, 'right') for ti, q in zip(a, ad)])
    rng = np.random.default_rng(seed); out = {k: np.empty(n, int) for k in KEYS}
    for i in range(n):
        c = PP[lo + (rng.random(len(a)) * (hi - lo)).astype(np.int64)]; bad = np.abs(c - a) < LEAD_MAX * NS
        for _ in range(20):
            if not bad.any():
                break
            c[bad] = PP[lo[bad] + (rng.random(int(bad.sum())) * (hi[bad] - lo[bad])).astype(np.int64)]; bad = np.abs(c - a) < LEAD_MAX * NS
        cc = counts(ev, np.sort(c))
        for k in KEYS:
            out[k][i] = cc[k]
    return out


def p1(null, obs):
    return (1 + int((null >= obs).sum())) / (len(null) + 1)


rows = []; t_ = time.time()
for ci, ((mname, dname), a) in enumerate(RUN.items()):
    ev = CATS[dname]['EV']; obs = counts(ev, a); st = statistics(ev, a)
    m = match_events(ev, a).merge(ev[['event_id', 'onset_1pct_utc']], on='event_id')
    rel = ((m.alarm_utc - m.onset_1pct_utc).dt.total_seconds() / 60) if len(m) else pd.Series(dtype=float)
    base = dict(method=mname, definition=dname, events=len(ev), alarms=len(a), **obs,
                share_of_caught_post_onset=obs['post'] / obs['caught'] if obs['caught'] else np.nan,
                median_alert_minus_onset_min=float(rel.median()) if len(rel) else np.nan, precision_caught=st['alert_precision'])
    nulls = {name: circ_null(ev, a, N_NULL, SEED + 10 * ci + j, band) for j, (name, band) in enumerate(BANDS.items())}
    nulls[f'regime-matched (+-{REGIME_WINDOW_D}d)'] = regime_null(ev, a, N_NULL, SEED + 10 * ci + 7)
    for nname, nl in nulls.items():
        for k in KEYS:
            rows.append(dict(**base, null=nname, endpoint=k, observed=obs[k], null_mean=float(nl[k].mean()), p_one_sided=p1(nl[k], obs[k])))
R = pd.DataFrame(rows); R.to_csv(OUT / '4_methods_definitions_nulls.csv', index=False)
print(f'nulls done in {time.time() - t_:.0f}s')
display(R.pivot_table(index=['definition', 'method'], columns=['null', 'endpoint'], values='p_one_sided').round(4))

## 5. Illusion table: what a standard evaluation would report versus the decomposition

In [ ]:
REG = f'regime-matched (+-{REGIME_WINDOW_D}d)'; PRI = 'circular >=7d (primary)'
ill = []
for (mname, dname), g in R.groupby(['method', 'definition']):
    def get(null, ep, col='p_one_sided'):
        return float(g[(g.null == null) & (g.endpoint == ep)][col].iloc[0])
    r0 = g.iloc[0]
    p_comp = get(PRI, 'caught'); p_pre = get(REG, 'pre'); p_post = get(REG, 'post')
    ill.append(dict(method=mname, definition=dname, events=int(r0.events), alarms=int(r0.alarms),
                    composite_caught=int(r0.caught), composite_p_primary=p_comp,
                    pre_onset=int(r0.pre), pre_null_mean_regime=get(REG, 'pre', 'null_mean'), pre_p_regime=p_pre, pre_p_4h1d=get('circular 4h-1d', 'pre'),
                    post_onset=int(r0.post), post_null_mean_regime=get(REG, 'post', 'null_mean'), post_p_regime=p_post,
                    onset_anchored_p_regime=get(REG, 'onset_anchored'),
                    share_of_caught_post_onset=float(r0.share_of_caught_post_onset), median_alert_minus_onset_min=float(r0.median_alert_minus_onset_min),
                    illusion=bool(p_comp < ALPHA and p_pre >= ALPHA and p_post < ALPHA),
                    reading=('detection masquerading as prediction' if (p_comp < ALPHA and p_pre >= ALPHA and p_post < ALPHA) else
                             'both pre- and post-onset skill' if (p_pre < ALPHA and p_post < ALPHA) else
                             'pre-onset only' if p_pre < ALPHA else 'post-onset only' if p_post < ALPHA else 'no skill')))
IL = pd.DataFrame(ill).sort_values(['definition', 'method']); IL.to_csv(OUT / '5_illusion_table.csv', index=False)
display(IL.round(4))

fig, ax = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
mlist = list(METHODS)
for i, dname in enumerate(DEFS):
    sub = IL[IL.definition == dname].set_index('method').reindex(mlist)
    x = np.arange(len(mlist))
    for off, col, lab, colr in [(-0.25, 'composite_p_primary', 'composite (standard evaluation)', '#7f8c8d'),
                                (0.0, 'pre_p_regime', 'pre-onset, regime-matched null', '#c0392b'),
                                (0.25, 'post_p_regime', 'post-onset, regime-matched null', '#2471a3')]:
        ax[i].bar(x + off, -np.log10(sub[col].to_numpy()), width=0.25, color=colr, label=lab)
    ax[i].axhline(-np.log10(ALPHA), color='k', ls='--', lw=1)
    ax[i].set_xticks(x); ax[i].set_xticklabels([m.split('_', 1)[1] for m in mlist], rotation=20, fontsize=9)
    ax[i].set_title(f'{dname} events (dev n={int(sub.events.iloc[0])})')
ax[0].set_ylabel('-log10 p (dashed: p = 0.05)'); ax[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / '5_illusion_by_method_definition.png', dpi=140); plt.show()

summary = dict(spec_sha256=SPEC_SHA, frozen_sha256=sha256_file(OUT / 'METHODS_FROZEN.json'),
               illusion_count=int(IL.illusion.sum()), configs=len(IL),
               readings=IL.groupby('reading').size().to_dict(), table=IL.to_dict('records'))
(OUT / 'METHODS_SUMMARY.json').write_text(json.dumps(summary, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))
print(f"illusion in {summary['illusion_count']} of {summary['configs']} method x definition configurations; readings: {summary['readings']}")